In [1]:
# Imports

import os
import pandas as pd
import numpy as np
import ray
from ray.rllib.algorithms.ppo import PPOConfig

import IPython.core.display_functions

from src.parsers import HMParser, CotevParser
from src.algorithms.rl import EnergyCommunityContributionPriorityV1

from src.utils import load_multiple_upacs_pv, iterate_resources, create_ppo_policies

import warnings
warnings.filterwarnings('ignore')

2025-02-06 11:41:59,596	WARNING deprecation.py:50 -- DeprecationWarning: `DirectStepOptimizer` has been deprecated. This will raise an error in the future!


In [2]:
# Data parsing

# EC data for non-renewable generators and batteries
data_ec = HMParser(file_path='/Users/ecgomes/DataspellProjects/pyecom/data/EC_V4.xlsx', ec_id=1)
data_ec.parse()

# EV data from the EV4EU simulator
data_ev = CotevParser(population_path=
                      '/Users/ecgomes/DataspellProjects/pyecom/data/simulation_dataframes_2years/population_731.csv',
                      driving_history_path=
                      '/Users/ecgomes/DataspellProjects/pyecom/data/simulation_dataframes_2years/ev_driving_history_731.csv',
                      assigned_segments_path='/Users/ecgomes/DataspellProjects/pyecom/data/simulation_dataframes_2years/assigned_segments_731.csv',
                      parse_date_start='2019',
                      parse_date_end='2020')
data_ev.parse()

# UPAC Data load
data_upacs = load_multiple_upacs_pv('/Users/ecgomes/Documents/PhD/UPAC data/upac*_pv.csv', resample='H')

In [3]:
# Create resources for the training environment

dataset_resources = iterate_resources(u=data_upacs, c=data_ec, e=data_ev, mode='monthly')

In [4]:
# Create the environment and check if everything is ok

temp_env = EnergyCommunityContributionPriorityV1(ren_generators=dataset_resources[
                                                                    list(dataset_resources.keys())[0]][:5],
                                                 generators=[],
                                                 loads=dataset_resources[list(dataset_resources.keys())[0]][5:10],
                                                 storages=dataset_resources[list(dataset_resources.keys())[0]][10:13],
                                                 evs=dataset_resources[list(dataset_resources.keys())[0]][13:-1],
                                                 aggregator=dataset_resources[list(dataset_resources.keys())[0]][-1],
                                                 storage_penalty=1,
                                                 ev_penalty=1,
                                                 balance_penalty=1)
temp_env.reset()
terminations = truncations = {a: False for a in temp_env.agents}
terminations['__all__'] = False
truncations['__all__'] = False
while not terminations['__all__'] and not truncations['__all__']:

    actions = temp_env.action_space_sample()
    next_obs, rewards, terminations, truncations, infos = temp_env.step(actions)

print('Terminated: {}'.format(terminations['__all__']))

Terminated: True


In [5]:
# Create the policies to train

# The keys of the dictionary respect the class names of the agents
gammas = {'Generator': 0.0, 'Storage': 0.9, 'Vehicle': 0.9, 'Aggregator': 0.9}

# Create the policies, one for each agent. Each policy has the name of the agent.
policies = create_ppo_policies(temp_env, gammas)

2025-02-06 11:42:07,176	WARNING algorithm_config.py:2578 -- Setting `exploration_config={}` because you set `_enable_rl_module_api=True`. When RLModule API are enabled, exploration_config can not be set. If you want to implement custom exploration behaviour, please modify the `forward_exploration` method of the RLModule at hand. On configs that have a default exploration config, this must be done with `config.exploration_config={}`.
2025-02-06 11:42:07,177	WARNING algorithm_config.py:2578 -- Setting `exploration_config={}` because you set `_enable_rl_module_api=True`. When RLModule API are enabled, exploration_config can not be set. If you want to implement custom exploration behaviour, please modify the `forward_exploration` method of the RLModule at hand. On configs that have a default exploration config, this must be done with `config.exploration_config={}`.
2025-02-06 11:42:07,177	WARNING algorithm_config.py:2578 -- Setting `exploration_config={}` because you set `_enable_rl_module

In [6]:
# Define the penalties

IMPORT_PENALTY = 1
EXPORT_PENALTY = 1
STORAGE_ACTION_PENALTY = 1
STORAGE_ACTION_REWARD = 5
EV_ACTION_PENALTY = 1
EV_ACTION_REWARD = 5
EV_REQUIREMENT_PENALTY = 2000
BALANCE_PENALTY = 5000

In [7]:
# Get the policy checkpoint - Sequential

from ray.tune import register_env
from ray.train import Checkpoint
from ray.rllib.algorithms.algorithm import Algorithm

ray.shutdown()
ray.init()

checkpoint_path = '/Users/ecgomes/ray_results/PPO_2025-02-05_22-03-35/PPO_EC_Contrib_V1_0bbc9_00000_0_2025-02-05_22-03-35/checkpoint_000000'

# We need to register the environment
temp_resources = dataset_resources['2019-01']
env = EnergyCommunityContributionPriorityV1(ren_generators=temp_resources[:5],
                                            generators=[],
                                            loads=temp_resources[5:10],
                                            storages=temp_resources[10:13],
                                            evs=temp_resources[13:-1],
                                            aggregator=temp_resources[-1],
                                            storage_penalty=STORAGE_ACTION_PENALTY,
                                            ev_penalty=EV_REQUIREMENT_PENALTY,
                                            balance_penalty=BALANCE_PENALTY)
register_env("EC_Contrib_V1", lambda config: env)

algo = Algorithm.from_checkpoint(checkpoint_path)

2025-02-06 11:42:17,979	INFO worker.py:1642 -- Started a local Ray instance.
2025-02-06 11:42:18,527	WARNING algorithm_config.py:2578 -- Setting `exploration_config={}` because you set `_enable_rl_module_api=True`. When RLModule API are enabled, exploration_config can not be set. If you want to implement custom exploration behaviour, please modify the `forward_exploration` method of the RLModule at hand. On configs that have a default exploration config, this must be done with `config.exploration_config={}`.
2025-02-06 11:42:18,529	WARNING algorithm_config.py:672 -- Cannot create PPOConfig from given `config_dict`! Property __stdout_file__ not supported.
2025-02-06 11:42:18,579	WARNING algorithm_config.py:2578 -- Setting `exploration_config={}` because you set `_enable_rl_module_api=True`. When RLModule API are enabled, exploration_config can not be set. If you want to implement custom exploration behaviour, please modify the `forward_exploration` method of the RLModule at hand. On con

In [8]:
# Run the learned policies

from copy import deepcopy

PATH = '../priority_paper_v1/'

current_storages = [bess.initial_charge for bess in dataset_resources['2019-01'][10:13]]
current_evs = [ev.initial_charge for ev in dataset_resources['2019-01'][13:-1]]

for i in list(dataset_resources.keys()):

    test_resources = deepcopy(dataset_resources[i])

    print('Day: {}'.format(i))

    for bess in np.arange(len(test_resources[10:13])):
        test_resources[10:13][bess].initial_charge = current_storages[bess]

    for ev in np.arange(len(test_resources[13:-1])):
        test_resources[13:-1][ev].initial_charge = current_evs[ev]


    test_env = EnergyCommunityContributionPriorityV1(ren_generators=test_resources[:5],
                                                     generators=[],
                                                     loads=test_resources[5:10],
                                                     storages=test_resources[10:13],
                                                     evs=test_resources[13:-1],
                                                     aggregator=test_resources[-1],
                                                     storage_penalty=STORAGE_ACTION_PENALTY,
                                                     ev_penalty=EV_REQUIREMENT_PENALTY,
                                                     balance_penalty=BALANCE_PENALTY)

    init_state = state = {a: algo.get_policy(a).get_initial_state() for a in test_env.agents.keys()}

    obs, info = test_env.reset()

    # Set up the terminations and truncations
    terminations = truncations = {a: False for a in test_env.agents}
    terminations['__all__'] = False
    truncations['__all__'] = False
    
    env_order = test_env.execution_order
    order_history = [env_order]

    while not terminations['__all__'] and not truncations['__all__']:

        current_agent = test_env.execution_order[test_env._current_agent_idx]

        action_dict, new_state, extra = algo.compute_single_action(observation=obs[current_agent],
                                                                   policy_id=current_agent,
                                                                   state=state[current_agent],
                                                                   explore=False)

        
        state[current_agent] = new_state

        # print(action_dict.keys())

        obs, rewards, terminations, truncations, info = test_env.step({current_agent: action_dict})
        
        if current_agent == 'aggregator' and not terminations['__all__'] and not truncations['__all__']:
            order_history.append(test_env.execution_order)

    current_storages = [bess.value[-1] for bess in test_env.storages]
    current_evs = [ev.value[-1] for ev in test_env.evs]

    pd_results = pd.DataFrame({})
    # Save the renewable generators
    for iter in np.arange(len(test_env.ren_generators)):
        agent_name = test_env.ren_generators[iter].name
        pd_results[agent_name] = test_env.ren_generators[iter].value

    # Save the storages
    for iter in np.arange(len(test_env.storages)):
        agent_name = test_env.storages[iter].name
        pd_results[agent_name] = test_env.storages[iter].value
        pd_results['{}_charge'.format(agent_name)] = test_env.storages[iter].charge
        pd_results['{}_discharge'.format(agent_name)] = test_env.storages[iter].discharge

    # Save the EVs
    for iter in np.arange(len(test_env.evs)):
        agent_name = test_env.evs[iter].name
        pd_results[agent_name] = test_env.evs[iter].value
        pd_results['{}_charge'.format(agent_name)] = test_env.evs[iter].charge
        pd_results['{}_discharge'.format(agent_name)] = test_env.evs[iter].discharge

    # Save the non-renewable generators
    #for iter in np.arange(len(seq_test_env.generators)):
    #    agent_name = seq_test_env.generators[iter].name
    #    pd_results[agent_name] = seq_test_env.generators[iter].value

    pd_results['imports'] = test_env.aggregator.imports
    pd_results['exports'] = test_env.aggregator.exports

    pd_results['energy_history'] = test_env.energy_history
    
    pd_results['order_history'] = order_history

    if not os.path.exists(PATH):
        os.makedirs(PATH)

    pd_results.to_csv('{}{}.csv'.format(PATH, i))

Day: 2019-01
Day: 2019-02
Day: 2019-03
Day: 2019-04
Day: 2019-05
Day: 2019-06
Day: 2019-07
Day: 2019-08
Day: 2019-09
Day: 2019-10
Day: 2019-11
Day: 2019-12
Day: 2020-01
Day: 2020-02
Day: 2020-03
Day: 2020-04
Day: 2020-05
Day: 2020-06
Day: 2020-07
Day: 2020-08
Day: 2020-09
Day: 2020-10
Day: 2020-11
Day: 2020-12
